# imports

In [1]:
import pandas as pd

# merge diffusion on the train split

In [2]:

all_subjects = pd.read_csv("all_subjects_tract_means.csv")
train_split = pd.read_csv("train_split.csv")

# keep only subjects that are in train_split
filtered = all_subjects[all_subjects["subjectID"].isin(train_split["Subject"])]

# add the train columns
merged = filtered.merge(
    train_split,
    left_on="subjectID",
    right_on="Subject",
    how="left"
)


merged = merged.drop(columns=["Subject"])

# double check counts

In [3]:
all_subjects["subjectID"].nunique()

1041

In [4]:
train_split["Subject"].nunique()

692

In [5]:
merged["subjectID"].nunique()

692

In [6]:
dd_df = pd.read_csv("DD_AUC_targets.csv")


In [7]:
dd_df["Subject"].nunique()

889

# Merge the uptaded train split on delay discounting dataframe

In [8]:


merged = merged.merge(
    dd_df[["Subject", "DDisc_AUC_40K"]],
    left_on="subjectID",
    right_on="Subject",
    how="left"
)

merged = merged.drop(columns=["Subject"])

# Re-order merged df

In [9]:
cols = ["subjectID", "age_group", "strata", "Gender", "DDisc_AUC_40K"]
merged = merged[cols + [c for c in merged.columns if c not in cols]]

In [10]:
diffusion_subjects_df = (
all_subjects[["subjectID"]]
.drop_duplicates()
.sort_values("subjectID")
.copy()
)

# Save a diffusion list subjects

In [11]:
diffusion_subjects_df.to_csv("diffusion_subjects.csv", index=False)

# Save the long format df with all relevant data

In [12]:
merged.to_csv("merged.csv", index=False)

# Make a wide format for proper analysis of diffusion data

In [13]:
subject_cols = [
    "subjectID",
    "age_group",
    "strata",
    "Gender",
    "DDisc_AUC_40K",
    "Release",
    "Acquisition",
    "Age",
    "3T_Full_MR_Compl",
    "7T_Full_MR_Compl",
    "MEG_FullProt_Compl",
]

wide = (
    merged
    .pivot_table(
        index=subject_cols,
        columns="tractID",
        values=["dki_fa", "dki_md", "dki_mk", "dki_awf"],
        aggfunc="first",
    )
)

wide.columns = [f"{tract}_{metric}" for metric, tract in wide.columns]
wide = wide.reset_index()

In [14]:
wide.to_csv("wide_diff_all_data.csv", index=False)